# LeetCode 2788 — Split Strings by Separator bằng Python

Link đề: https://leetcode.com/problems/split-strings-by-separator/ · Độ khó: **Easy** · Chủ đề: Array, String

Skeleton LeetCode:

```python
class Solution:
    def splitWordsBySeparator(self, words: List[str], separator: str) -> List[str]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **tách** từng chuỗi trong `words` theo ký tự `separator`, **bỏ các chuỗi rỗng**, giữ **đúng thứ tự** ban đầu, rồi gộp mọi mảnh vào một danh sách.
2. Nắm chắc `str.split(sep)`, điều khác biệt giữa `split(sep)` và `split()` (không đối số), tính **"truthy / falsy"** của chuỗi rỗng, `append` so với `extend`, và `str.join`.
3. Đi từ **duyệt từng ký tự bằng tay** (không dùng `split`) → **`split` + lọc rỗng** → **`extend` / `chain`** → **mẹo `join` rồi `split` một lần** → **regex với `re.escape`**.
4. Hiểu vì sao các ký tự `. | $` là **ký tự đặc biệt trong regex** và phải `re.escape` nếu dùng `re.split`.
5. Tránh các lỗi: quên lọc chuỗi rỗng, dùng `split()` không đối số, dùng `append` thay vì `extend` (thành danh sách lồng nhau), và truyền separator thô vào regex.

## 2. Đề bài & ví dụ

Cho mảng các chuỗi `words` và một ký tự `separator`. **Tách** mỗi chuỗi trong `words` theo `separator`.

Trả về một mảng các chuỗi chứa **các chuỗi mới được tạo sau khi tách**, **không gồm chuỗi rỗng**.

**Lưu ý:**

- `separator` chỉ dùng để xác định chỗ cần tách, **không** nằm trong các chuỗi kết quả.
- Một lần tách có thể cho **nhiều hơn hai** chuỗi.
- Các chuỗi kết quả phải giữ **cùng thứ tự** như ban đầu (thứ tự các `words`, và thứ tự các mảnh trong từng từ).

**Ví dụ 1**

- Input: `words = ["one.two.three","four.five","six"]`, `separator = "."`
- Output: `["one","two","three","four","five","six"]`
- Giải thích: tách `"one.two.three"` thành `"one"`, `"two"`, `"three"`; `"four.five"` thành `"four"`, `"five"`; `"six"` không có dấu chấm nên giữ nguyên.

**Ví dụ 2**

- Input: `words = ["$easy$","$problem$"]`, `separator = "$"`
- Output: `["easy","problem"]`
- Giải thích: tách `"$easy$"` cho `""`, `"easy"`, `""`; bỏ các chuỗi rỗng. Tương tự với `"$problem$"`.

**Ví dụ 3**

- Input: `words = ["|||"]`, `separator = "|"`
- Output: `[]`
- Giải thích: tách `"|||"` ra toàn chuỗi rỗng, bỏ hết nên được mảng rỗng.

Constraints:

- `1 <= words.length <= 100`
- `1 <= words[i].length <= 20`
- Các ký tự của `words[i]` là chữ cái tiếng Anh viết thường hoặc các ký tự trong `".,|$#@"` (không gồm dấu ngoặc kép).
- `separator` là một ký tự thuộc `".,|$#@"`.

In [1]:
words = ["one.two.three", "four.five", "six"]
sep = "."
for w in words:
    print(repr(w), "->", w.split(sep))

'one.two.three' -> ['one', 'two', 'three']
'four.five' -> ['four', 'five']
'six' -> ['six']


## 3. Kiến thức nền

### 3.1. `str.split(sep)`: tách chuỗi theo một ký tự

`"a.b.c".split(".")` trả về danh sách `['a', 'b', 'c']`. Quy tắc quan trọng: khi hai dấu tách đứng **liền nhau**, hoặc dấu tách ở **đầu / cuối** chuỗi, kết quả chứa **chuỗi rỗng `''`**.

In [2]:
print("a.b.c".split("."))      # ['a', 'b', 'c']
print("a..b".split("."))       # ['a', '', 'b']      <- hai dấu liền nhau
print("$easy$".split("$"))     # ['', 'easy', '']    <- dấu ở đầu và cuối
print("|||".split("|"))        # ['', '', '', '']    <- n dấu cho n + 1 mảnh
print("six".split("."))        # ['six']             <- không có dấu thì giữ nguyên

['a', 'b', 'c']
['a', '', 'b']
['', 'easy', '']
['', '', '', '']
['six']


### 3.2. `split()` **không đối số** là việc khác!

`s.split()` (không truyền gì) tách theo **khoảng trắng** và **tự bỏ chuỗi rỗng**. Nó **không** tách theo dấu `.`. Ở bài này ta phải truyền `separator` rõ ràng: `s.split(separator)`.

In [3]:
print("a b  c".split())        # ['a', 'b', 'c']   (tách theo khoảng trắng, bỏ rỗng)
print("a.b.c".split())         # ['a.b.c']          (không tách gì cả!)
print("a b  c".split(" "))     # ['a', 'b', '', 'c']

['a', 'b', 'c']
['a.b.c']
['a', 'b', '', 'c']


### 3.3. Chuỗi rỗng là "falsy"

Trong `if s:`, chuỗi rỗng `''` được coi như `False`, chuỗi khác rỗng là `True`. Nhờ vậy `[p for p in parts if p]` **lọc bỏ chuỗi rỗng** rất gọn.

In [4]:
print(bool(""), bool("a"), bool("  "))      # False True True
parts = ["", "easy", ""]
print([p for p in parts if p])                # ['easy']
print(list(filter(None, parts)))              # ['easy']  (filter(None, ...) giữ phần tử truthy)

False True True
['easy']
['easy']


### 3.4. `append` khác `extend`

- `res.append(x)` thêm **một** phần tử `x` (nếu `x` là danh sách thì thành danh sách lồng).
- `res.extend(iterable)` thêm **từng** phần tử của iterable vào `res`.

In [5]:
res = []
res.append(["a", "b"])
print(res)                 # [['a', 'b']]  (danh sách lồng)
res = []
res.extend(["a", "b"])
print(res)                 # ['a', 'b']

[['a', 'b']]
['a', 'b']


### 3.5. Duyệt từng ký tự và nối chuỗi

`for ch in s:` duyệt từng ký tự. Muốn dựng một mảnh ta nối `cur += ch` hoặc gom vào danh sách rồi `"".join(...)`. Khi gặp ký tự tách ta "chốt" mảnh hiện tại.

In [6]:
s = "ab.c"
for ch in s:
    print(repr(ch), end=" ")      # 'a' 'b' '.' 'c'
print()
print("".join(["a", "b"]), "-".join(["x", "y", "z"]))   # ab x-y-z

'a' 'b' '.' 'c' 
ab x-y-z


### 3.6. Ký tự đặc biệt trong regex và `re.escape`

Module `re` (biểu thức chính quy) dùng `. | $` với nghĩa **đặc biệt**: `.` khớp mọi ký tự, `|` là "hoặc", `$` là "cuối chuỗi". Muốn khớp **chính ký tự đó**, ta dùng `re.escape(sep)` (thêm dấu `\` thích hợp).

In [7]:
import re
print(re.escape("."), re.escape("|"), re.escape("$"), re.escape("#"))
print(re.split(".", "a.b"))                # ['', '', '', '']  <- '.' khớp MỌI ký tự (sai ý)
print(re.split(re.escape("."), "a.b"))     # ['a', 'b']

\. \| \$ \#
['', '', '', '']
['a', 'b']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy nghĩ như **cắt dây**: mỗi từ là một sợi dây, mỗi ký tự `separator` là một chỗ cắt. Sau khi cắt, ta nhặt các **đoạn không rỗng** theo thứ tự rồi bỏ vào giỏ chung.

Với `"$easy$"` và `separator = "$"`:

```
$ e a s y $
↑         ↑
cắt       cắt     =>  '' , 'easy' , ''   =>  bỏ rỗng  =>  'easy'
```

Với `"one.two.three"`: cắt tại hai dấu chấm ⇒ `one`, `two`, `three`.

**Quan sát:** việc tách một từ **không phụ thuộc** các từ khác, nên bài toán lặp lại cùng một thao tác cho từng từ rồi **ghép kết quả theo thứ tự**.

In [8]:
for w in ["one.two.three", "$easy$", "|||", "six"]:
    sep = w[0] if w[0] in "$|" else "."
    pieces = w.split(sep)
    print(f"{w!r:18} tách theo {sep!r}: {pieces}  -> bỏ rỗng: {[p for p in pieces if p]}")

'one.two.three'    tách theo '.': ['one', 'two', 'three']  -> bỏ rỗng: ['one', 'two', 'three']
'$easy$'           tách theo '$': ['', 'easy', '']  -> bỏ rỗng: ['easy']
'|||'              tách theo '|': ['', '', '', '']  -> bỏ rỗng: []
'six'              tách theo '.': ['six']  -> bỏ rỗng: ['six']


## 5. Cách 1 — Brute Force / baseline: duyệt từng ký tự, không dùng `split`

**Ý tưởng:** với mỗi từ, đi qua từng ký tự:

1. Nếu ký tự **khác** `separator`: ghép vào mảnh hiện tại `cur`.
2. Nếu ký tự **bằng** `separator`: nếu `cur` **không rỗng** thì thêm vào kết quả, rồi đặt lại `cur = ""`.
3. Hết từ: nếu `cur` còn dữ liệu thì thêm vào kết quả (mảnh cuối không có dấu tách phía sau).

Cách này tự cài `split` bằng tay, đúng yêu cầu "bỏ chuỗi rỗng" bằng cách chỉ thêm `cur` khi nó không rỗng.

In [9]:
from typing import List

class SolutionScan:
    def splitWordsBySeparator(self, words: List[str], separator: str) -> List[str]:
        result = []
        for w in words:
            cur = ""
            for ch in w:
                if ch == separator:
                    if cur:
                        result.append(cur)
                    cur = ""
                else:
                    cur += ch
            if cur:
                result.append(cur)
        return result

In [10]:
sol = SolutionScan()
print(sol.splitWordsBySeparator(["one.two.three", "four.five", "six"], "."))   # ['one', 'two', 'three', 'four', 'five', 'six']
print(sol.splitWordsBySeparator(["$easy$", "$problem$"], "$"))                  # ['easy', 'problem']
print(sol.splitWordsBySeparator(["|||"], "|"))                                  # []

['one', 'two', 'three', 'four', 'five', 'six']
['easy', 'problem']
[]


**Độ phức tạp:** gọi `L` là tổng độ dài các chuỗi. Mỗi ký tự được xét một lần; nối chuỗi `cur += ch` tốn tối đa theo độ dài mảnh (rất nhỏ, `<= 20`) ⇒ Time `O(L)`, Space `O(L)` cho kết quả.

**Nút thắt:** ta đang **tự cài lại** thao tác mà Python đã có sẵn (`str.split`). Cách này giúp hiểu bản chất nhưng dài; trong thực tế dùng `split` rồi lọc rỗng.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — dùng `split` rồi lọc rỗng.** `w.split(separator)` đã cho các mảnh (kể cả rỗng); `[p for p in w.split(separator) if p]` loại các mảnh rỗng. Gộp cho mọi từ bằng một biểu thức hai tầng.

**Đột phá 2 — `extend` thay `append`.** Mỗi từ cho **một danh sách** mảnh; dùng `result.extend(...)` để trải phẳng, không tạo danh sách lồng.

**Đột phá 3 — mẹo "nối rồi tách một lần".** Nếu nối mọi từ bằng chính `separator` (`separator.join(words)`) rồi `split(separator)` **một lần**, ta được cùng các mảnh như tách riêng từng từ: ranh giới giữa hai từ giờ **là một dấu tách**, nên không có mảnh nào bị dính hai từ vào nhau. Sau đó chỉ cần lọc rỗng.

**Đột phá 4 — regex phải thoát ký tự.** `re.split(re.escape(separator), w)` làm cùng việc; nhưng nếu quên `re.escape` thì `.`, `|`, `$` có nghĩa đặc biệt và kết quả sai.

In [11]:
words = ["one.two.three", "four.five", "six"]
sep = "."
print(sep.join(words))
print(sep.join(words).split(sep))
print([p for p in sep.join(words).split(sep) if p])

one.two.three.four.five.six
['one', 'two', 'three', 'four', 'five', 'six']
['one', 'two', 'three', 'four', 'five', 'six']


## 7. Các cách tiếp theo

### Cách 2 — `split` + lọc rỗng bằng biểu thức hai tầng (code nộp)

**Từng đoạn:**

- Tầng ngoài: `for w in words` (giữ thứ tự từ).
- Tầng trong: `for p in w.split(separator)` (giữ thứ tự mảnh).
- Điều kiện `if p` giữ mảnh không rỗng.

In [12]:
class SolutionSplit:
    def splitWordsBySeparator(self, words: List[str], separator: str) -> List[str]:
        return [p for w in words for p in w.split(separator) if p]

In [13]:
sol = SolutionSplit()
print(sol.splitWordsBySeparator(["one.two.three", "four.five", "six"], "."))
print(sol.splitWordsBySeparator(["$easy$", "$problem$"], "$"))
print(sol.splitWordsBySeparator(["|||"], "|"))

['one', 'two', 'three', 'four', 'five', 'six']
['easy', 'problem']
[]


**Lưu ý đọc biểu thức hai tầng:** thứ tự các `for` trong comprehension giống như hai vòng lồng nhau viết từ ngoài vào trong: `for w in words` rồi `for p in w.split(separator)`.

**Độ phức tạp:** Time `O(L)`, Space `O(L)`.

### Cách 3 — Vòng lặp thường với `extend` / `filter` (dễ đọc cho người mới)

**Từng đoạn:** dùng vòng `for` ngoài từng từ; mỗi từ `extend` danh sách mảnh đã lọc bằng `filter(None, ...)` (giữ mảnh truthy). Cách này tách việc "tách" và "lọc" thành hai bước rõ ràng.

In [14]:
class SolutionExtend:
    def splitWordsBySeparator(self, words: List[str], separator: str) -> List[str]:
        result = []
        for w in words:
            parts = w.split(separator)          # có thể chứa chuỗi rỗng
            result.extend(filter(None, parts))   # bỏ chuỗi rỗng, trải phẳng vào result
        return result

In [15]:
sol = SolutionExtend()
print(sol.splitWordsBySeparator(["one.two.three", "four.five", "six"], "."))
print(sol.splitWordsBySeparator(["$easy$", "$problem$"], "$"))
print(sol.splitWordsBySeparator(["|||"], "|"))

['one', 'two', 'three', 'four', 'five', 'six']
['easy', 'problem']
[]


**Độ phức tạp:** Time `O(L)`, Space `O(L)`.

### Cách 4 — Mẹo nối bằng `separator` rồi `split` một lần

**Từng đoạn:**

- `big = separator.join(words)`: nối các từ bằng chính `separator` ⇒ một chuỗi dài.
- `big.split(separator)`: mọi mảnh của mọi từ, và các ranh giới từ cũng là chỗ tách.
- Lọc rỗng.

**Vì sao đúng?** Giữa hai từ liền kề ta chèn đúng một `separator`; mảnh cuối của từ trước và mảnh đầu của từ sau bị **ngăn cách** bởi dấu đó, nên không bị dính. Nếu mảnh cuối của từ trước rỗng (từ kết thúc bằng dấu tách) thì ta chỉ có thêm một mảnh rỗng, bị lọc đi.

In [16]:
class SolutionJoin:
    def splitWordsBySeparator(self, words: List[str], separator: str) -> List[str]:
        return [p for p in separator.join(words).split(separator) if p]

In [17]:
sol = SolutionJoin()
print(sol.splitWordsBySeparator(["one.two.three", "four.five", "six"], "."))
print(sol.splitWordsBySeparator(["$easy$", "$problem$"], "$"))
print(sol.splitWordsBySeparator(["|||"], "|"))

['one', 'two', 'three', 'four', 'five', 'six']
['easy', 'problem']
[]


**Độ phức tạp:** Time `O(L)`, Space `O(L)` (chuỗi nối thêm `O(L)`).

### Cách 5 — Regex `re.split` với `re.escape`

**Từng đoạn:** tạo mẫu `pattern = re.escape(separator)` để coi `separator` là **ký tự thường**; `re.split(pattern, w)` tách giống `w.split(separator)`; lọc rỗng.

In [18]:
import re

class SolutionRegex:
    def splitWordsBySeparator(self, words: List[str], separator: str) -> List[str]:
        pattern = re.escape(separator)
        return [p for w in words for p in re.split(pattern, w) if p]

In [19]:
sol = SolutionRegex()
print(sol.splitWordsBySeparator(["one.two.three", "four.five", "six"], "."))
print(sol.splitWordsBySeparator(["$easy$", "$problem$"], "$"))
print(sol.splitWordsBySeparator(["|||"], "|"))

['one', 'two', 'three', 'four', 'five', 'six']
['easy', 'problem']
[]


**Độ phức tạp:** Time `O(L)` (với mẫu một ký tự), Space `O(L)`. Chậm hơn `str.split` một chút vì phải biên dịch / chạy máy regex; ở bài này không cần regex, ta dùng để biết bẫy ký tự đặc biệt.

## 8. TRACE từng bước

Trace Cách 1 (duyệt ký tự) trên `words = ["$easy$", "ab.c"]`, `separator = "$"`:

Từ `"$easy$"`:

| ký tự | `cur` trước | hành động | `cur` sau | `result` |
|---|---|---|---|---|
| `$` | `""` | là dấu tách, `cur` rỗng ⇒ không thêm | `""` | `[]` |
| `e` | `""` | nối | `"e"` | `[]` |
| `a` | `"e"` | nối | `"ea"` | `[]` |
| `s` | `"ea"` | nối | `"eas"` | `[]` |
| `y` | `"eas"` | nối | `"easy"` | `[]` |
| `$` | `"easy"` | dấu tách, `cur` khác rỗng ⇒ thêm `"easy"` | `""` | `["easy"]` |

Hết từ, `cur` rỗng ⇒ không thêm. Từ `"ab.c"` (không có `$`): nối hết thành `"ab.c"`, cuối từ thêm vào ⇒ `["easy", "ab.c"]`. Chú ý `.` ở đây là ký tự thường vì `separator` là `$`.

In [20]:
def trace_scan(words, separator):
    result = []
    for w in words:
        print(f"== từ {w!r}")
        cur = ""
        for ch in w:
            before = cur
            action = ""
            if ch == separator:
                if cur:
                    result.append(cur)
                    action = f"chốt {cur!r}"
                else:
                    action = "dấu tách, mảnh rỗng -> bỏ"
                cur = ""
            else:
                cur += ch
                action = "nối"
            print(f"  ch={ch!r}: cur {before!r} -> {cur!r} ({action}), result={result}")
        if cur:
            result.append(cur)
            print(f"  hết từ: chốt {cur!r}, result={result}")
    return result

trace_scan(["$easy$", "ab.c"], "$")

== từ '$easy$'
  ch='$': cur '' -> '' (dấu tách, mảnh rỗng -> bỏ), result=[]
  ch='e': cur '' -> 'e' (nối), result=[]
  ch='a': cur 'e' -> 'ea' (nối), result=[]
  ch='s': cur 'ea' -> 'eas' (nối), result=[]
  ch='y': cur 'eas' -> 'easy' (nối), result=[]
  ch='$': cur 'easy' -> '' (chốt 'easy'), result=['easy']
== từ 'ab.c'
  ch='a': cur '' -> 'a' (nối), result=['easy']
  ch='b': cur 'a' -> 'ab' (nối), result=['easy']
  ch='.': cur 'ab' -> 'ab.' (nối), result=['easy']
  ch='c': cur 'ab.' -> 'ab.c' (nối), result=['easy']
  hết từ: chốt 'ab.c', result=['easy', 'ab.c']


['easy', 'ab.c']

In [21]:
def trace_split(words, separator):
    result = []
    for w in words:
        parts = w.split(separator)
        kept = [p for p in parts if p]
        print(f"{w!r:18} split -> {parts}  giữ -> {kept}")
        result.extend(kept)
    print("kết quả:", result)
    return result

trace_split(["one.two.three", "four.five", "six"], ".")
trace_split(["|||", "a||b"], "|")

'one.two.three'    split -> ['one', 'two', 'three']  giữ -> ['one', 'two', 'three']
'four.five'        split -> ['four', 'five']  giữ -> ['four', 'five']
'six'              split -> ['six']  giữ -> ['six']
kết quả: ['one', 'two', 'three', 'four', 'five', 'six']
'|||'              split -> ['', '', '', '']  giữ -> []
'a||b'             split -> ['a', '', 'b']  giữ -> ['a', 'b']
kết quả: ['a', 'b']


['a', 'b']

## 9. Code nộp LeetCode

Dùng Cách 2 (`split` + lọc rỗng).

In [22]:
from typing import List

class Solution:
    def splitWordsBySeparator(self, words: List[str], separator: str) -> List[str]:
        # với mỗi từ (giữ thứ tự), tách theo separator rồi giữ các mảnh không rỗng
        return [part for w in words for part in w.split(separator) if part]

In [23]:
print(Solution().splitWordsBySeparator(["one.two.three", "four.five", "six"], "."))
print(Solution().splitWordsBySeparator(["$easy$", "$problem$"], "$"))
print(Solution().splitWordsBySeparator(["|||"], "|"))

['one', 'two', 'three', 'four', 'five', 'six']
['easy', 'problem']
[]


## 10. Vì sao thuật toán đúng

**Cách 2.** `w.split(separator)` với `separator` là một ký tự cho đúng các đoạn nằm **giữa** các lần xuất hiện của `separator` trong `w` (kể cả đoạn rỗng): nếu `w` chứa `k` dấu tách thì được `k + 1` đoạn, và các đoạn **không chứa** `separator`. Lọc `if part` bỏ đúng các đoạn rỗng. Hai vòng `for` duyệt các từ theo thứ tự rồi các đoạn theo thứ tự, nên kết quả đúng thứ tự yêu cầu.

**Cách 1.** Xét một từ. Bất biến: `cur` là mảnh hiện tại chưa chốt, chỉ gồm các ký tự khác `separator` đứng sau dấu tách gần nhất. Khi gặp `separator` ta chốt `cur` (nếu không rỗng) rồi làm rỗng; cuối từ chốt mảnh cuối nếu không rỗng. Vậy kết quả là các đoạn không rỗng giữa các dấu tách, giống Cách 2.

**Cách 4 (nối rồi tách).** Gọi `s = separator.join(words)`. Các dấu tách trong `s` gồm các dấu bên trong từng từ cộng các dấu chèn giữa hai từ liền kề. Các đoạn không rỗng của `s` là: các đoạn không rỗng của từ 1, của từ 2, ..., theo thứ tự, vì ranh giới giữa hai từ là một dấu tách nên không đoạn nào vắt ngang hai từ. Điều này đúng vì `separator` là **một ký tự** và các từ không chứa gì ngoài ký tự bình thường và dấu tách. Do đó kết quả giống Cách 2.

**Cách 5.** `re.escape(separator)` làm cho `separator` được hiểu là ký tự thường; `re.split` trên một ký tự cố định tách đúng giống `str.split`.

Test mục 12 kiểm chứng bằng thực nghiệm cùng các tính chất bổ sung: không có chuỗi rỗng, không chứa `separator`, nối lại các mảnh bằng bỏ `separator` cho cùng nội dung ban đầu.

## 11. Phân tích Time / Space Complexity

`L` là tổng độ dài của tất cả các chuỗi trong `words`; `m` là số mảnh kết quả.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Duyệt từng ký tự | `O(L)` | `O(L)` | tự cài đặt, nhiều dòng |
| 2. `split` + lọc | `O(L)` | `O(L)` | **code nộp**, ngắn nhất |
| 3. `extend(filter(...))` | `O(L)` | `O(L)` | dễ đọc cho người mới |
| 4. Nối rồi tách một lần | `O(L)` | `O(L)` | cần thêm chuỗi trung gian `O(L)` |
| 5. Regex | `O(L)` | `O(L)` | hằng số lớn hơn, cần `re.escape` |

Với `words.length <= 100`, `words[i].length <= 20` thì `L <= 2000`, mọi cách đều tức thì.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: ba ví dụ của đề; từ không có dấu tách; từ chỉ gồm dấu tách; dấu ở đầu, ở cuối, liền nhau; từ một ký tự; mọi ký tự đặc biệt `. , | $ # @` làm separator (kiểm tra các ký tự regex đặc biệt).

In [24]:
sols = [SolutionScan(), SolutionSplit(), SolutionExtend(), SolutionJoin(), SolutionRegex(), Solution()]
cases = [
    ("ví dụ 1", ["one.two.three", "four.five", "six"], ".", ["one", "two", "three", "four", "five", "six"]),
    ("ví dụ 2", ["$easy$", "$problem$"], "$", ["easy", "problem"]),
    ("ví dụ 3", ["|||"], "|", []),
    ("không có dấu tách", ["abc", "de"], ",", ["abc", "de"]),
    ("một ký tự là dấu tách", ["#"], "#", []),
    ("một ký tự thường", ["a"], "#", ["a"]),
    ("dấu liền nhau", ["a..b"], ".", ["a", "b"]),
    ("dấu đầu và cuối", [",ab,cd,"], ",", ["ab", "cd"]),
    ("separator regex đặc biệt '|'", ["a|b|c", "d"], "|", ["a", "b", "c", "d"]),
    ("separator regex đặc biệt '$'", ["a$b", "$$", "c$"], "$", ["a", "b", "c"]),
    ("separator '@'", ["x@y@z"], "@", ["x", "y", "z"]),
    ("dấu khác không bị tách", ["a.b,c"], ",", ["a.b", "c"]),
    ("dấu '.' trong từ, tách theo '|'", ["a.b|c.d"], "|", ["a.b", "c.d"]),
    ("từ rỗng sau tách ở giữa", ["a,,,b", ",,,"], ",", ["a", "b"]),
]
for name, words, sep, exp in cases:
    for s in sols:
        got = s.splitWordsBySeparator(words[:], sep)
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> ['one', 'two', 'three', 'four', 'five', 'six']
PASS ví dụ 2 -> ['easy', 'problem']
PASS ví dụ 3 -> []
PASS không có dấu tách -> ['abc', 'de']
PASS một ký tự là dấu tách -> []
PASS một ký tự thường -> ['a']
PASS dấu liền nhau -> ['a', 'b']
PASS dấu đầu và cuối -> ['ab', 'cd']
PASS separator regex đặc biệt '|' -> ['a', 'b', 'c', 'd']
PASS separator regex đặc biệt '$' -> ['a', 'b', 'c']
PASS separator '@' -> ['x', 'y', 'z']
PASS dấu khác không bị tách -> ['a.b', 'c']
PASS dấu '.' trong từ, tách theo '|' -> ['a.b', 'c.d']
PASS từ rỗng sau tách ở giữa -> ['a', 'b']


### 12.2. Phản ví dụ: `re.split` không `escape`

Với separator `"."` hay `"|"` hay `"$"`, `re.split(separator, w)` **không** cho kết quả mong muốn. Ta kiểm tra lỗi để thấy test của ta sẽ bắt được nếu ai viết regex sai.

In [25]:
def sai_regex(words, separator):
    return [p for w in words for p in re.split(separator, w) if p]       # sai: không escape

for words, sep in [(["a.b"], "."), (["a|b"], "|"), (["a$b"], "$")]:
    mong_muon = Solution().splitWordsBySeparator(words, sep)
    thuc_te = sai_regex(words, sep)
    print(f"sep={sep!r}: đúng={mong_muon}, regex không escape={thuc_te}")
    assert thuc_te != mong_muon

sep='.': đúng=['a', 'b'], regex không escape=[]
sep='|': đúng=['a', 'b'], regex không escape=['a', '|', 'b']
sep='$': đúng=['a', 'b'], regex không escape=['a$b']


### 12.3. Đối chiếu ngẫu nhiên (hợp lệ theo đề)

Sinh `words` với `1 <= len <= 100`, mỗi từ dài `1..20`, ký tự là chữ thường `a, b, c` hoặc ký tự trong `".,|$#@"`; `separator` ngẫu nhiên trong `".,|$#@"`. Để có nhiều dấu tách, ta chọn tỉ lệ ký tự tách cao. So sánh 6 cách với nhau và kiểm các tính chất của kết quả.

In [26]:
import random
random.seed(2788)
SEPS = ".,|$#@"

def gen_case():
    k = random.randint(1, 100)
    sep = random.choice(SEPS)
    p_sep = random.choice([0.1, 0.3, 0.6, 0.9])
    words = []
    for _ in range(k):
        L = random.randint(1, 20)
        w = "".join(sep if random.random() < p_sep else random.choice("abc" + SEPS.replace(sep, ""))
                    for _ in range(L))
        words.append(w)
    return words, sep

for _ in range(600):
    words, sep = gen_case()
    assert 1 <= len(words) <= 100 and all(1 <= len(w) <= 20 for w in words)
    assert all(ch in "abcdefghijklmnopqrstuvwxyz" + SEPS for w in words for ch in w)
    exp = SolutionScan().splitWordsBySeparator(words[:], sep)
    for s in sols:
        got = s.splitWordsBySeparator(words[:], sep)
        assert got == exp, (type(s).__name__, words, sep, got, exp)
    # các tính chất của kết quả
    assert all(p != "" and sep not in p for p in exp)
    assert "".join(exp) == "".join(w.replace(sep, "") for w in words)       # giữ nội dung và thứ tự
    snap = words[:]
    Solution().splitWordsBySeparator(words, sep)
    assert words == snap
print("600 ca ngẫu nhiên: 6 cách khớp nhau, kết quả có đủ tính chất")

600 ca ngẫu nhiên: 6 cách khớp nhau, kết quả có đủ tính chất


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên lọc chuỗi rỗng

In [27]:
def sai_khong_loc(words: List[str], separator: str) -> List[str]:
    return [p for w in words for p in w.split(separator)]       # sai: giữ cả ''

print("sai :", sai_khong_loc(["$easy$", "$problem$"], "$"))              # ['', 'easy', '', '', 'problem', '']
print("đúng:", Solution().splitWordsBySeparator(["$easy$", "$problem$"], "$"))     # ['easy', 'problem']

sai : ['', 'easy', '', '', 'problem', '']
đúng: ['easy', 'problem']


**Vì sao sai:** `"$easy$".split("$")` cho `['', 'easy', '']`. Đề yêu cầu **không** gồm chuỗi rỗng ⇒ phải lọc.

### Lỗi 2: Dùng `split()` không đối số

In [28]:
def sai_split_khong_doi_so(words: List[str], separator: str) -> List[str]:
    return [p for w in words for p in w.split() if p]            # sai: tách theo khoảng trắng

print("sai :", sai_split_khong_doi_so(["one.two.three"], "."))         # ['one.two.three']
print("đúng:", Solution().splitWordsBySeparator(["one.two.three"], "."))

sai : ['one.two.three']
đúng: ['one', 'two', 'three']


**Vì sao sai:** `w.split()` tách theo **khoảng trắng**, không phải theo `separator`. Phải truyền `separator` làm đối số: `w.split(separator)`.

### Lỗi 3: Dùng `append` thay vì `extend` (danh sách lồng)

In [29]:
def sai_append(words: List[str], separator: str) -> List[str]:
    result = []
    for w in words:
        result.append([p for p in w.split(separator) if p])     # sai: thêm CẢ danh sách như một phần tử
    return result

print("sai :", sai_append(["one.two", "three"], "."))                 # [['one', 'two'], ['three']]
print("đúng:", Solution().splitWordsBySeparator(["one.two", "three"], "."))     # ['one', 'two', 'three']

sai : [['one', 'two'], ['three']]
đúng: ['one', 'two', 'three']


**Vì sao sai:** đề muốn một danh sách **phẳng** các chuỗi. `append(list)` tạo danh sách lồng; dùng `extend` hoặc biểu thức hai tầng.

### Lỗi 4: Truyền `separator` thô vào regex

In [30]:
def sai_regex_chua_escape(words: List[str], separator: str) -> List[str]:
    return [p for w in words for p in re.split(separator, w) if p]

print("sai :", sai_regex_chua_escape(["one.two.three"], "."))       # []  ('.' khớp mọi ký tự)
print("đúng:", Solution().splitWordsBySeparator(["one.two.three"], "."))

sai : []
đúng: ['one', 'two', 'three']


**Vì sao sai:** trong regex, `.` khớp **mọi** ký tự nên mọi ký tự thành chỗ tách ⇒ toàn mảnh rỗng ⇒ kết quả rỗng. Với `|` hoặc `$` cũng có nghĩa đặc biệt. Dùng `re.escape(separator)` hoặc tốt hơn là `str.split`.

### Lỗi 5: Dùng `strip` thay cho lọc rỗng

In [31]:
def sai_strip(words: List[str], separator: str) -> List[str]:
    return [p for w in words for p in w.strip(separator).split(separator)]    # sai: dấu liền nhau vẫn sinh ''

print("sai :", sai_strip(["a..b", "|||"], "."), sai_strip(["|||"], "|"))     # có chuỗi rỗng
print("đúng:", Solution().splitWordsBySeparator(["a..b", "|||"], "."))

sai : ['a', '', 'b', '|||'] ['']
đúng: ['a', 'b', '|||']


**Vì sao sai:** `strip` chỉ bỏ dấu ở **hai đầu**, không xử lý các dấu **liền nhau ở giữa** (`"a..b"` vẫn cho `''`), và với chuỗi toàn dấu tách như `"|||"` vẫn còn `''` sau `split`. Cách chắc chắn là **lọc** `if part`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tách chuỗi theo ký tự rồi **bỏ rỗng**" → `[p for p in s.split(sep) if p]`. Nhớ `split(sep)` **giữ** chuỗi rỗng, `split()` **bỏ** nhưng tách theo khoảng trắng.
- "Làm cùng việc cho mỗi phần tử rồi trải phẳng" → biểu thức hai tầng `[y for x in xs for y in f(x)]` hoặc `extend`.
- Ký tự đặc biệt của regex: `. ^ $ * + ? { } [ ] \ | ( )`. Nếu bắt buộc dùng regex hãy `re.escape`.
- Với dữ liệu nhỏ, ưu tiên hàm có sẵn (`split`), ít lỗi hơn tự cài đặt.
- Test luôn gồm: không có dấu tách, toàn dấu tách, dấu đầu / cuối / liền nhau, từng ký tự đặc biệt.

## 15. Biến thể của bài

1. **Giữ chuỗi rỗng** (bỏ bước lọc): kết quả là `[p for w in words for p in w.split(sep)]`.
2. **Tách theo nhiều ký tự** (một tập `seps`): dùng `re.split("[" + re.escape("".join(seps)) + "]", w)` hoặc thay các `seps` bằng một ký tự chung rồi `split`.
3. **Tách theo một chuỗi dài** (separator nhiều ký tự): `w.split(separator)` vẫn dùng được; mẹo nối rồi tách cần cẩn thận ranh giới.
4. **Chỉ tách tối đa `k` lần:** `w.split(sep, k)`.
5. **Trả về số lượng mảnh** hoặc **mảnh dài nhất**: `len(result)`, `max(result, key=len)`.
6. **Đảo thứ tự các mảnh** hoặc **đảo từng mảnh** (như bài 151, 557): kết hợp `[::-1]`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt ký tự | tự chốt mảnh khi gặp dấu tách | `O(L)` | `O(L)` | hiểu bản chất, không có `split` |
| 2. `split` + lọc | biểu thức hai tầng | `O(L)` | `O(L)` | **code nộp** |
| 3. `extend(filter(...))` | vòng lặp ngoài, trải phẳng | `O(L)` | `O(L)` | dễ đọc cho người mới |
| 4. Nối rồi tách | `separator.join(words).split(...)` | `O(L)` | `O(L)` | ngắn nhất, cần hiểu lý do |
| 5. Regex | `re.split(re.escape(sep), w)` | `O(L)` | `O(L)` | khi dấu tách phức tạp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"a,,b".split(",")`, `"a b  c".split()`, `[p for p in ["", "x", ""] if p]`, `["a", "b"] + ["c"]` và `[["a"], ["b"]]` sau `extend`/`append`.

**Bài 2 (tính tay).** Với `words = ["a#b##c", "#d#"]`, `separator = "#"`: viết `split` của từng từ, sau lọc rồi kết quả cuối.

**Bài 3 (điền TODO).** Hoàn thành `split_words_todo(words, sep)`.

In [32]:
def split_words_todo(words, sep):
    result = []
    for w in words:
        # TODO 1: parts = w.split(sep)
        # TODO 2: thêm vào result các phần tử không rỗng của parts (dùng extend hoặc vòng for)
        pass
    return result

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `split_scan(words, sep)` **không dùng `split`**, chỉ duyệt từng ký tự (Cách 1). Kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `split_keep_empty(words, sep)` giữ chuỗi rỗng. (b) Viết `split_multi(words, seps)` tách theo **nhiều** ký tự (`seps` là chuỗi như `".,"`), bỏ rỗng, **không** dùng regex.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `['a', '', 'b']`; `['a', 'b', 'c']`; `['x']`; sau `extend(["a","b"])` là `['a','b']`, sau `append(["a","b"])` là `[['a','b']]`. Bài 2: `"a#b##c".split("#") = ['a','b','','c']`; `"#d#".split("#") = ['','d','']`.
- **Mức 2 (vừa):** Bài 3: `parts = w.split(sep)`; `result.extend(p for p in parts if p)`. Bài 4: giữ `cur`, chốt khi gặp `sep` (nếu `cur` không rỗng), chốt nốt cuối từ.
- **Mức 3 (sâu):** Bài 5(a): bỏ bước lọc: `result.extend(w.split(sep))`. Bài 5(b): trong vòng ký tự, điều kiện chốt là `ch in seps` thay cho `ch == sep`; hoặc thay mọi ký tự trong `seps` bằng cùng một ký tự (ví dụ `seps[0]`) rồi `split`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert "a,,b".split(",") == ["a", "", "b"]
assert "a b  c".split() == ["a", "b", "c"]
assert [p for p in ["", "x", ""] if p] == ["x"]
lst = []
lst.extend(["a", "b"])
assert lst == ["a", "b"]
lst2 = []
lst2.append(["a", "b"])
assert lst2 == [["a", "b"]]

# Bài 2
assert "a#b##c".split("#") == ["a", "b", "", "c"]
assert "#d#".split("#") == ["", "d", ""]
assert Solution().splitWordsBySeparator(["a#b##c", "#d#"], "#") == ["a", "b", "c", "d"]

# Bài 3
def split_words_todo_ans(words, sep):
    result = []
    for w in words:
        parts = w.split(sep)
        result.extend(p for p in parts if p)
    return result

assert split_words_todo_ans(["one.two.three", "four.five", "six"], ".") == ["one", "two", "three", "four", "five", "six"]
assert split_words_todo_ans(["$easy$", "$problem$"], "$") == ["easy", "problem"]
assert split_words_todo_ans(["|||"], "|") == []

# Bài 4
def split_scan(words, sep):
    result = []
    for w in words:
        cur = ""
        for ch in w:
            if ch == sep:
                if cur:
                    result.append(cur)
                cur = ""
            else:
                cur += ch
        if cur:
            result.append(cur)
    return result

assert split_scan(["one.two.three", "four.five", "six"], ".") == ["one", "two", "three", "four", "five", "six"]
assert split_scan(["$easy$", "$problem$"], "$") == ["easy", "problem"]
assert split_scan(["|||"], "|") == []

# Bài 5(a)
def split_keep_empty(words, sep):
    result = []
    for w in words:
        result.extend(w.split(sep))
    return result

assert split_keep_empty(["$easy$"], "$") == ["", "easy", ""]
assert split_keep_empty(["|||"], "|") == ["", "", "", ""]

# Bài 5(b): nhiều ký tự tách, không regex
def split_multi(words, seps):
    result = []
    for w in words:
        cur = ""
        for ch in w:
            if ch in seps:
                if cur:
                    result.append(cur)
                cur = ""
            else:
                cur += ch
        if cur:
            result.append(cur)
    return result

assert split_multi(["a.b,c", ".,"], ".,") == ["a", "b", "c"]
assert split_multi(["x|y$z"], "|$") == ["x", "y", "z"]
random.seed(4)
for _ in range(300):
    words, sep = gen_case()
    assert split_scan(words, sep) == Solution().splitWordsBySeparator(words, sep)
    assert split_multi(words, sep) == Solution().splitWordsBySeparator(words, sep)
    assert [p for p in split_keep_empty(words, sep) if p] == Solution().splitWordsBySeparator(words, sep)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (làm cùng việc cho từng phần tử rồi trải phẳng; lọc rỗng)

```python
from typing import List

class Solution:
    def splitWordsBySeparator(self, words: List[str], separator: str) -> List[str]:
        return [part for w in words for part in w.split(separator) if part]
```

**Checklist**

- [ ] Dùng `w.split(separator)` (truyền đối số), **không** phải `w.split()`?
- [ ] Đã **lọc chuỗi rỗng** (`if part`)?
- [ ] Kết quả là danh sách **phẳng** (dùng `extend` / biểu thức hai tầng, không `append` danh sách)?
- [ ] Giữ **thứ tự** các từ và các mảnh?
- [ ] Nếu dùng regex: đã `re.escape(separator)` (vì `.`, `|`, `$` là ký tự đặc biệt)?
- [ ] Đã thử từ không có dấu tách, toàn dấu tách (`"|||"`), dấu ở đầu / cuối / liền nhau?
- [ ] Signature đúng `splitWordsBySeparator(self, words: List[str], separator: str) -> List[str]` và có `from typing import List`?